In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

### root+verbimuster -> count mustrid tabel 'alati' ja 'mitte kunagi' verbide jaoks

In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()

In [19]:
query = """
SELECT * FROM transactions_verbs_obl_kohakaandes
limit 20
"""

s = pd.read_sql_query(query, con)
s

,head_id,verb,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,kaane
0,2,toimuma,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in
1,3,saama,7,keel,obl,S,"all,com,pl",UNK,UNK,all
2,10,tulema,19,sina,obl,P,"ad,sg",UNK,YES,ad
3,11,viilima,22,tund,obl,S,"com,el,pl",UNK,UNK,el
4,11,viilima,23,juht,obl,S,"ad,com,sg",UNK,YES,ad
5,25,muutuma,40,mis,obl,P,"el,sg",UNK,UNK,el
6,33,minema,59,rahvas,obl,S,"all,com,sg",UNK,UNK,all
7,37,tekkima,69,see,obl,P,"el,sg",UNK,UNK,el
8,51,kutsuma,85,elu,obl,S,"adit,com,sg",UNK,UNK,adit
9,53,tulema,88,toim,obl,S,"adit,com,sg",UNK,UNK,adit


In [3]:
query = """
SELECT *
FROM patterns_transaction_isikud_alati
limit 20
"""
s = pd.read_sql_query(query, con)
s

,head_id,pat_id,transaction_id,phrase_nr,verb_word,root_word,pat_deprel,word_deprel,pos,phrase_case,tr_feats
0,1053,1,1803,1,küsima,kaart,obl,obl,S,abl,"abl,com,pl"
1,2878,1,4893,1,küsima,mina,obl,obl,P,abl,"abl,sg"
2,3956,1,6832,1,küsima,ise,obl,obl,P,abl,"abl,sg"
3,4240,1,7369,1,küsima,teineteise,obl,obl,P,abl,"abl,sg"
4,11575,1,21070,1,küsima,füsioterapeut,obl,obl,S,abl,"abl,com,sg"
5,12634,1,23359,1,küsima,ise,obl,obl,P,abl,"abl,sg"
6,14389,1,26599,1,küsima,Vahur,obl,obl,S,abl,"abl,prop,sg"
7,17242,1,31847,1,küsima,mina,obl,obl,P,abl,"abl,sg"
8,17364,1,32042,1,küsima,Ülo,obl,obl,S,abl,"abl,prop,sg"
9,17906,1,32933,1,küsima,isa,obl,obl,S,abl,"abl,com,sg"


# ##############################################################
# alati root+verb+kaane+head_cnt

In [57]:
query = """
SELECT root_word, verb_word||'_'||phrase_case as verb_pat, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_alati
group by root_word, verb_pat
order by head_cnt desc
"""

sb = pd.read_sql_query(query, con)
sb

,root_word,verb_pat,head_cnt
0,mina,meeldima_all,24503
1,mina,olema_ad,12961
2,aasta,saama_ad,10707
3,mina,tulema_ad,9122
4,hääletus,panema_all,8666
...,...,...,...
267816,žüriiliige,minema_all,1
267817,žüriiliige,pakkuma_all,1
267818,žüriiliige,teenima_abl,1
267819,žüriiliige,tulema_all,1


### mis on top 100 mustrit ja kui palju roote need ära katavad

In [59]:
verb_pats100 = list(sb[:100]["verb_pat"])
verb_pats100roots = list(set(list(sb[:100]["root_word"])))

In [60]:
len(verb_pats100roots)

37

In [64]:
len(list(set(list(sb["root_word"]))))

72281

# alati root + verb_pat count

In [20]:
query = """

select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat--, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_alati
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc

"""
source = pd.read_sql_query(query, con)

In [21]:
source

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


## alati top 100 root

In [6]:
list(source.iloc[:100]["root_word"])

['tema',
 'mina',
 'sina',
 'kes',
 'ise',
 'see',
 'inimene',
 'mees',
 'teine',
 'naine',
 'laps',
 'keegi',
 'kõik',
 'rahvas',
 'mis',
 'riik',
 'sõber',
 'maa',
 'ema',
 'Venemaa',
 'ajakirjanik',
 'eestlane',
 'juht',
 'poiss',
 'Eesti',
 'tee',
 'klient',
 'üksteise',
 'isa',
 'firma',
 'valitsus',
 'oma',
 'president',
 'üks',
 'linn',
 'tüdruk',
 'töötaja',
 'töö',
 'tänav',
 'politsei',
 'külaline',
 'noor',
 'auto',
 'pool',
 'vanem',
 'omanik',
 'liige',
 'politseinik',
 'õpilane',
 'tütar',
 'publik',
 'soomlane',
 'õpetaja',
 'vend',
 'kolleeg',
 'vaataja',
 'noormees',
 'koht',
 'elanik',
 'lugeja',
 'kodanik',
 'turg',
 'abikaasa',
 'poeg',
 'pere',
 'isik',
 'arst',
 'teineteise',
 'lava',
 'pank',
 'iseenese',
 'Saksamaa',
 'meeskond',
 'ettevõte',
 'ametnik',
 'ameeriklane',
 'maailm',
 'küsimus',
 'võim',
 'saar',
 'ala',
 'venelane',
 'jõud',
 'vastane',
 'naaber',
 'ostja',
 'näitleja',
 'poja',
 'minister',
 'kaaslane',
 'autor',
 'tuttav',
 'peaminister',
 'mäng

## alati top 100 mustrite arvu alusel elusad

### precision = 75-87% on elusad
kui lubada teine,maa,firma, töö, politsei,maailm, pank, võim, ettevõte, jõud,turg,linn,

### elusad ka riigid: 
tema, mina, sina, kes, ise, mees, inimene, naine, laps, keegi, rahvas, sõber, riik, ajakirjanik, juht, ema, eestlane, Venemaa, poiss, klient, Eesti, üksteise, isa, president, valitsus, oma, külaline, töötaja, tüdruk, vanem, noor, publik, politseinik, liigem omanik, õpilane, vend, tütar, soomlane, kolleeg, õpetaja, vaataja, lugeja, noormees, elanik, teineteise, kodanik, abikaasa, poeg, pere, arst, isik, iseenese, ameeriklane, meeskond, ametnik, Saksamaa, venelane, ostja, kaaslane, minister, vastane, naaber, näitleja, poja, peremees, peaminister, autor, mängija, poliitik, kohtunik, tuttav, ohver, liider, tudeng

### mis ei ole elus ega ka agent või on kahtlane: 
see,kõik,mis,tee,üks,tänav,auto,pool,koht,küsimus,lava,saar,ala, 

teine,maa,firma, töö, politsei,maailm, pank, võim, ettevõte, jõud,turg,linn,

# alati verbi esinemistele vastav rootide arv

### st kui palju on roote (root_word) kui verb_count on 1, 2, 3, 4 jne

In [7]:
df1 = pd.DataFrame(source.groupby('verb_count')['root_word'].nunique())
df1

,root_word
verb_count,
1,44945
2,9281
3,4452
4,2598
5,1779
...,...
269,1
282,1
301,1


In [8]:
# kui palju roote esineb ainult 1 mustris: 44945
source[source["verb_count"]==1]

,root_word,verb_count
27336,žüriihääletus,1
27337,žurnalistikatudeng,1
27338,žukov,1
27339,žtaalia,1
27340,žnternationa,1
...,...,...
72276,0-millennium,1
72277,"0,5",1
72278,-6%,1
72279,-2%,1


In [9]:
# kui palju roote esineb vähemalt 10 mustris: 5,517
source[source["verb_count"]>=10]

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
5512,Angela,10
5513,Andre,10
5514,Aleksejeva,10
5515,AGA,10


In [10]:
# kui palju roote esineb vähemalt 100 mustris: 111
source[source["verb_count"]>=100]

,root_word,verb_count
0,tema,402
1,mina,398
2,sina,301
3,kes,282
4,ise,269
...,...,...
106,iga,101
107,avalikkus,101
108,tudeng,100
109,loom,100


In [65]:
# verb_pats100roots = top100 sagedasema verb+kääne rootid
roots_atleast100 = list(set(list(source[source["verb_count"]>=100]["root_word"])))

In [67]:
yhisosa = []

for w in verb_pats100roots:
    if w in roots_atleast100:
        yhisosa.append(w)

In [68]:
len(yhisosa)

14

# alati juursõnade sagedusi obl fraasides üldse kui mustreid==1

In [34]:
one_pat_roots = list(source[source["verb_count"]==1]["root_word"])

onepatlist = ",".join(["'"+r+"'" for r in one_pat_roots])


query = """
SELECT root_word, count(distinct head_id) as head_cnt 
FROM transactions_verbs_obl_kohakaandes
where root_word in ({rootlist})
group by root_word
order by head_cnt desc
""".format(rootlist=onepatlist)

s = pd.read_sql_query(query, con)
s

,root_word,head_cnt
0,veerandfinaal,3154
1,küll,1959
2,maikuu,1745
3,käsutus,1633
4,Ülikool,1413
...,...,...
44940,024,1
44941,02-mees,1
44942,0.450,1
44943,0-määr,1


In [37]:
s[s["head_cnt"]>=100]

,root_word,head_cnt
0,veerandfinaal,3154
1,küll,1959
2,maikuu,1745
3,käsutus,1633
4,Ülikool,1413
...,...,...
252,sadamateater,101
253,rongiõnnetus,101
254,kehastus,101
255,raadioeeter,100


# ##############################################################
# vahel root+verb+kaane+head_cnt

In [75]:
query = """
SELECT root_word, verb_word||'_'||phrase_case as verb_pat, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_vahel_2
group by root_word, verb_pat
order by head_cnt desc
"""

sb = pd.read_sql_query(query, con)
sb

,root_word,verb_pat,head_cnt
0,käsi,saama_adit,13487
1,mina,olema_ad,12961
2,see,saama_el,10873
3,aasta,saama_ad,10707
4,mina,tulema_ad,9122
...,...,...,...
861234,žüriisõel,pääsema_el,1
861235,žüris,kuuluma_adit,1
861236,α-aminohappejääk,koosnema_el,1
861237,ω-3-rasvhape,moodustama_el,1


### mis on top 100 mustrit ja kui palju roote need ära katavad

In [76]:
verb_pats100 = list(sb[:100]["verb_pat"])
verb_pats100roots = list(set(list(sb[:100]["root_word"])))

In [77]:
len(verb_pats100roots)

33

In [78]:
len(list(set(list(sb["root_word"]))))

157030

# vahel root + verb_pat count

In [79]:
query = """

select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat--, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_vahel_2
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc

"""
source = pd.read_sql_query(query, con)

In [80]:
source

,root_word,verb_count
0,tema,1520
1,see,1389
2,mina,1298
3,ise,1079
4,mis,1050
...,...,...
157025,0%,1
157026,-9%,1
157027,+-255,1
157028,%-see,1


## vahel top 100 root

In [81]:
list(source.iloc[:100]["root_word"])

['tema',
 'see',
 'mina',
 'ise',
 'mis',
 'kes',
 'inimene',
 'sina',
 'teine',
 'riik',
 'mees',
 'maa',
 'laps',
 'Eesti',
 'koht',
 'linn',
 'naine',
 'aasta',
 'töö',
 'kõik',
 'aeg',
 'päev',
 'eestlane',
 'Venemaa',
 'auto',
 'tänav',
 'tee',
 'Tallinn',
 'pool',
 'oma',
 'firma',
 'kool',
 'maja',
 'koha',
 'juht',
 'elu',
 'osa',
 'turg',
 'maailm',
 'liige',
 'külg',
 'mäng',
 'valitsus',
 'pank',
 'kodu',
 'üks',
 'käsi',
 'poiss',
 'küsimus',
 'rahvas',
 'noor',
 'kord',
 'lõpp',
 'sõna',
 'keegi',
 'ala',
 'ettevõte',
 'töötaja',
 'klient',
 'film',
 'asi',
 'seadus',
 'saar',
 'meeskond',
 'algus',
 'pere',
 'jõud',
 'ema',
 'Saksamaa',
 'silm',
 'plats',
 'tüdruk',
 'elanik',
 'sõber',
 'piir',
 'president',
 'MM',
 'õpilane',
 'olukord',
 'laud',
 'isik',
 'võistlus',
 'politsei',
 'lava',
 'tase',
 'nägu',
 'laev',
 'omanik',
 'liit',
 'kodanik',
 'üksteise',
 'otsus',
 'jalg',
 'kohtumine',
 'kuu',
 'pea',
 'keel',
 'meri',
 'suvi',
 'ameeriklane']

## alati top 100 mustrite arvu alusel elusad

### precision = 


### elusad ka riigid: 


### mis ei ole elus ega ka agent või on kahtlane: 



# vahel verbi esinemistele vastav rootide arv

### st kui palju on roote (root_word) kui verb_count on 1, 2, 3, 4 jne

In [82]:
df1 = pd.DataFrame(source.groupby('verb_count')['root_word'].nunique())
df1

,root_word
verb_count,
1,96877
2,19673
3,9144
4,5481
5,3747
...,...
1050,1
1079,1
1298,1


In [83]:
# kui palju roote esineb ainult 1 mustris: 96.877
source[source["verb_count"]==1]

,root_word,verb_count
60153,ω-linoleenhape,1
60154,ω-3-rasvhape,1
60155,α-aminohappejääk,1
60156,žüris,1
60157,žüriisõel,1
...,...,...
157025,0%,1
157026,-9%,1
157027,+-255,1
157028,%-see,1


In [84]:
# kui palju roote esineb vähemalt 10 mustris: 14.309
source[source["verb_count"]>=10]

,root_word,verb_count
0,tema,1520
1,see,1389
2,mina,1298
3,ise,1079
4,mis,1050
...,...,...
14304,7,10
14305,60,10
14306,40aastane,10
14307,30%,10


In [85]:
# kui palju roote esineb vähemalt 100 mustris: 1223
source[source["verb_count"]>=100]

,root_word,verb_count
0,tema,1520
1,see,1389
2,mina,1298
3,ise,1079
4,mis,1050
...,...,...
1218,finaal,100
1219,bussijuht,100
1220,astumine,100
1221,aktsiaturg,100


In [86]:
# verb_pats100roots = top100 sagedasema verb+kääne rootid
roots_atleast100 = list(set(list(source[source["verb_count"]>=100]["root_word"])))

In [87]:
yhisosa = []

for w in verb_pats100roots:
    if w in roots_atleast100:
        yhisosa.append(w)

In [88]:
len(yhisosa)

27

# vahel juursõnade sagedusi obl fraasides üldse kui mustreid==1

In [89]:
one_pat_roots = list(source[source["verb_count"]==1]["root_word"])

onepatlist = ",".join(["'"+r+"'" for r in one_pat_roots])


query = """
SELECT root_word, count(distinct head_id) as head_cnt 
FROM transactions_verbs_obl_kohakaandes
where root_word in ({rootlist})
group by root_word
order by head_cnt desc
""".format(rootlist=onepatlist)

s = pd.read_sql_query(query, con)
s

,root_word,head_cnt
0,üldkokkuvõte,242
1,erandkord,214
2,jänn,164
3,McDonald,163
4,silmaots,132
...,...,...
96872,"0,28",1
96873,"0,2%",1
96874,"0,0",1
96875,+-255,1


In [90]:
s[s["head_cnt"]>=100]

,root_word,head_cnt
0,üldkokkuvõte,242
1,erandkord,214
2,jänn,164
3,McDonald,163
4,silmaots,132
5,eelisjärjekord,125
6,vasaraheide,124
7,uju,124
8,täismaht,119
9,narkouim,116


hist1 = source.hist()
source.plot.hist(column=["verb_count"], by="root_word", figsize=(10, 8))

words = list(source["root_word"])
counts = list(source["verb_count"])

plt.bar(words, counts)
plt.title("alati verbide esinemine koos rootsõnaga")
plt.ylabel("verb count")
plt.xlabel("root")
plt.show()

# ##############################################################
# mitte_kunagi root+verb+kaane+head_cnt

In [61]:
query = """
SELECT root_word, verb_word||'_'||phrase_case as verb_pat, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_mitte_kunagi_2
group by root_word, verb_pat
order by head_cnt desc
"""

sb2 = pd.read_sql_query(query, con)
sb2

,root_word,verb_pat,head_cnt
0,mina,meeldima_all,24503
1,käsi,saama_adit,13487
2,mina,olema_ad,12961
3,see,saama_el,10873
4,aasta,saama_ad,10707
...,...,...,...
1399669,žüriisõel,pääsema_el,1
1399670,žüris,kuuluma_adit,1
1399671,ˇkuida,panema_in,1
1399672,β-glükaan-solubilaa,osalema_in,1


### mis on top 100 mustrit ja kui palju roote need ära katavad

In [62]:
verb_pats100_2 = list(sb2[:100]["verb_pat"])
verb_pats100roots2 = list(set(list(sb2[:100]["root_word"])))

In [63]:
len(verb_pats100roots2)

36

# mitte kunagi root+verb count

In [11]:
query = """

select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat--, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_mitte_kunagi_2
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc

"""
source2 = pd.read_sql_query(query, con)

In [12]:
source2

,root_word,verb_count
0,aasta,1947
1,mis,1881
2,see,1803
3,tema,1682
4,maa,1682
...,...,...
205229,%-põhimõte,1
205230,%-ilis,1
205231,$1,1
205232,$-esi,1


# mitte kunagi top 100 root

In [13]:
list(source2.iloc[:100]["root_word"])

['aasta',
 'mis',
 'see',
 'tema',
 'maa',
 'aeg',
 'mina',
 'päev',
 'sõna',
 'Eesti',
 'koht',
 'lõpp',
 'teine',
 'tänav',
 'riik',
 'kord',
 'linn',
 'Tallinn',
 'pool',
 'juht',
 'kodu',
 'maja',
 'nädal',
 'osa',
 'jaanuar',
 'koha',
 'algus',
 'töö',
 'september',
 'auto',
 'Tartu',
 'tee',
 'aprill',
 'august',
 'oktoober',
 'ise',
 'õhtu',
 'juuni',
 'märts',
 'kool',
 'kuu',
 'november',
 'veebruar',
 'hetk',
 'detsember',
 'mäng',
 'üks',
 'suvi',
 'maailm',
 'sina',
 'kohtumine',
 'juuli',
 'elu',
 'käsi',
 'Venemaa',
 'Euroopa',
 'kes',
 'piir',
 'hommik',
 'inimene',
 'rida',
 'mai',
 'teade',
 'mõte',
 'Moskva',
 'andmed',
 'ala',
 'vald',
 'silm',
 'hinnang',
 'tingimus',
 'väide',
 'piirkond',
 'sügis',
 'Soome',
 'teema',
 'suund',
 'USA',
 'mägi',
 'lava',
 'iga',
 'plats',
 'meri',
 'rand',
 'film',
 'asi',
 'tuba',
 'keel',
 'saal',
 'ring',
 'paik',
 'olukord',
 'vesi',
 'võistlus',
 'turg',
 'Saksamaa',
 'küla',
 'öö',
 'pank',
 'küsimus']

## mitte kunagi top 100 mustrite arvu alusel elusad

### precision = 15% on elusad


### elusad ka riigid: 
mina, tema, juht, Eesti, sina, Venemaa, Euroopa, ise, Soome, kes, USA, käsi, inimene, silm, Saksamaa

### mis ei ole elus ega ka agent või on kahtlane: 

aeg/timex: aasta,päev,hetk, õhtu,nädal, suvi, hommik,kord, algus,sügis,jaanuar, reede, aprill, september,kuu,  esmaspäev,veebruar, oktoober, märts,juuni, november, kevad,detsember, august, neljapöev, pühapäev, nädalavahetus, juuli,teisipäev, kolmapäev,öö, kevade, tulevik,mai, 


koht: Tallinn, koht, linn, kodu,Tartu,riik, maailm,laupäev,maja,koha,kool,  ala,vald, Moskva, lava, piirkond, saal,
turg,



aeg, sõna, mis, see, lõpp, maa, tänav, andmed, hinnang, väide, teine, teade, tee, mõte, pool, osa, mood, kinnitus, töö, elu, komme, põhjus, määr, üks, kohtumine, auto, mäng, viis, piir, tingimus, teema, tase, olukord

# mitte kunagi root arv iga verbimustri sageduse jaoks

In [14]:
df2 = pd.DataFrame(source2.groupby('verb_count')['root_word'].nunique())
df2

,root_word
verb_count,
1,125339
2,25621
3,11909
4,7150
5,4884
...,...
1670,1
1682,2
1803,1


In [15]:
# kui palju roote esineb ainult 1 mustris: 125,339
source2[source2["verb_count"]==1]

,root_word,verb_count
79895,ω-3-rasvhape,1
79896,β-glükaan-solubilaa,1
79897,ˇkuida,1
79898,žüris,1
79899,žüriisõel,1
...,...,...
205229,%-põhimõte,1
205230,%-ilis,1
205231,$1,1
205232,$-esi,1


In [16]:
# kui palju roote esineb vähemalt 10 mustris: 20,120
source2[source2["verb_count"]>=10]

,root_word,verb_count
0,aasta,1947
1,mis,1881
2,see,1803
3,tema,1682
4,maa,1682
...,...,...
20115,7,10
20116,30aastane,10
20117,2000.,10
20118,13,10


In [17]:
# kui palju roote esineb vähemalt 100 mustris: 2222
source2[source2["verb_count"]>=100]

,root_word,verb_count
0,aasta,1947
1,mis,1881
2,see,1803
3,tema,1682
4,maa,1682
...,...,...
2217,koosviibimine,100
2218,keskpank,100
2219,font,100
2220,blokk,100


# mitte kunagi juursõnade sagedusi obl fraasides üldse kui mustreid==1

In [39]:
#one_pat_roots = list(source2[source2["verb_count"]==1]["root_word"])

#onepatlist = ",".join(["'"+r+"'" for r in one_pat_roots])


query = """
SELECT root_word, count(distinct head_id) as head_cnt 
FROM transactions_verbs_obl_kohakaandes
where root_word in 

(
select root_word from (select root_word, count(verb_pat) as verb_count from (
SELECT root_word, verb_word||'_'||phrase_case as verb_pat --, count(distinct head_id) as head_cnt 
FROM patterns_transaction_isikud_mitte_kunagi_2
group by root_word, verb_pat
--order by head_cnt desc
) as tbl1
group by root_word
order by verb_count desc) as tbl2
where verb_count==1
)

group by root_word
order by head_cnt desc
"""#.format(rootlist=onepatlist)

s = pd.read_sql_query(query, con)
s

,root_word,head_cnt
0,ksülaan,69
1,obligatsiooniomanik,35
2,häirenupp,28
3,tervishoiunõue,22
4,Kolbakov,21
...,...,...
125334,%-põhimõte,1
125335,%-ilis,1
125336,$1,1
125337,$-esi,1


# ##############################################################
# tabel verb+kääne+root count + annotated t/f

In [49]:
s3 = pd.read_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage_v2.csv", sep=";", encoding="utf-8", )
s3

,verb,kaane,root_count,annotated
0,saama,el,21132,True
1,andma,all,12526,True
2,rääkima,el,10930,True
3,jääma,el,9936,True
4,tulema,ad,9166,True
...,...,...,...,...
30081,šveitsima,el,1,False
30082,švipsima,ad,1,False
30083,žestikuleerima,ad,1,False
30084,žisraelima,ad,1,False


In [56]:
s3[(s3["annotated"]==False) & (s3["root_count"]>=100)]

,verb,kaane,root_count,annotated
1564,tõmbuma,el,238,False
1776,hiilima,el,202,False
1813,kargama,el,197,False
1962,pistma,el,178,False
2174,pressima,el,155,False
...,...,...,...,...
2966,petma,in,101,False
2970,trügima,in,101,False
2977,kaevama,el,100,False
2980,kukutama,in,100,False


In [69]:
s3[s3["annotated"]==True]

,verb,kaane,root_count,annotated
0,saama,el,21132,True
1,andma,all,12526,True
2,rääkima,el,10930,True
3,jääma,el,9936,True
4,tulema,ad,9166,True
...,...,...,...,...
13615,väärama,all,5,True
13626,võõrustama,abl,5,True
13636,ärkama,abl,5,True
13637,ärritama,abl,5,True


In [71]:
s3[(s3["annotated"]==True) & (s3["root_count"]<10)]

,verb,kaane,root_count,annotated
10215,aeglustuma,abl,9,True
10216,aeguma,abl,9,True
10218,agiteerima,all,9,True
10219,ahvatlema,ill,9,True
10220,ajendama,abl,9,True
...,...,...,...,...
13615,väärama,all,5,True
13626,võõrustama,abl,5,True
13636,ärkama,abl,5,True
13637,ärritama,abl,5,True


### mis on top 100 mustrit ja kui palju roote need ära katavad

In [73]:
verb_pats100_3 = list(sb3[:100]["verb_pat"])
verb_pats100roots3 = list(set(list(sb3[:100]["root_word"])))

In [74]:
len(verb_pats100roots3)

33

In [3]:
con.close()